# Plot the gene-panel comparison in Jupyter


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# Find the project folder by looking upward from Jupyter's current folder.
here = Path.cwd().resolve()
PROJECT = next((p for p in [here, *here.parents] if (p / 'outputs' / 'scripts' / '05_compare_panel_with_meropenem.py').exists()), None)
if PROJECT is None:
    raise FileNotFoundError("Open Jupyter in this project folder, or one of its subfolders.")
SUMMARY = PROJECT / 'work' / 'data' / 'processed' / 'panel_vs_meropenem_summary.csv'
OUT = PROJECT / 'outputs' / 'figures'
OUT.mkdir(parents=True, exist_ok=True)
print("Project folder:", PROJECT)
print("Comparison summary exists:", SUMMARY.exists())

In [ ]:
summary = pd.read_csv(SUMMARY, encoding='utf-8-sig')

def get_count(label):
    row = summary.loc[summary['measure'].eq(label)]
    if row.empty:
        raise ValueError(f'Missing count in summary: {label}')
    return int(row.iloc[0]['numerator'])

# Table rows are AST results (R, S); columns are gene-panel result (positive, negative).
tp = get_count('True positive (R and panel positive)')
fn = get_count('False negative (R and panel negative)')
fp = get_count('False positive (S and panel positive)')
tn = get_count('True negative (S and panel negative)')
counts = [[tp, fn], [fp, tn]]
print('Resistant row:', counts[0], '| total =', sum(counts[0]))
print('Susceptible row:', counts[1], '| total =', sum(counts[1]))

In [ ]:
# A clean 2×2 heatmap: green = matching category; amber = mismatch.
fig, ax = plt.subplots(figsize=(9, 6.2), constrained_layout=True)
cmap = ListedColormap(['#e7f3ec', '#fff1d6'])
match_map = [[0, 1], [1, 0]]
ax.imshow(match_map, cmap=cmap, vmin=0, vmax=1, aspect='auto')

ax.set_xticks([0, 1], labels=['Panel positive', 'Panel negative'], fontsize=11)
ax.set_yticks([0, 1], labels=['Resistant (R)', 'Susceptible (S)'], fontsize=11)
ax.xaxis.tick_top()
ax.tick_params(axis='both', length=0, pad=12)

row_totals = [sum(counts[0]), sum(counts[1])]
for i in range(2):
    for j in range(2):
        n = counts[i][j]
        pct = 100 * n / row_totals[i]
        tag = 'MATCH' if i == j else 'MISMATCH'
        ax.text(j, i - 0.08, f'{n:,}', ha='center', va='center',
                fontsize=25, fontweight='bold', color='#17332a')
        ax.text(j, i + 0.18, f'{pct:.1f}% of this AST row  ·  {tag}',
                ha='center', va='center', fontsize=10, color='#344b43')

ax.set_title('Selected carbapenemase gene panel vs. meropenem result',
             fontsize=16, fontweight='bold', pad=26)
ax.set_xlabel('NCBI gene panel call', fontsize=11, labelpad=16)
ax.set_ylabel('Submitted meropenem phenotype', fontsize=11, labelpad=14)
ax.set_xticks([0.5], minor=True)
ax.set_yticks([0.5], minor=True)
ax.grid(which='minor', color='white', linewidth=3)
ax.tick_params(which='minor', bottom=False, left=False)
for spine in ax.spines.values():
    spine.set_visible(False)

fig.text(0.5, -0.015,
         'Panel positive = at least one selected blaKPC, blaNDM, or selected OXA-48-like call marked COMPLETE. '
         'Intermediate isolates (n=55) excluded.',
         ha='center', fontsize=9, color='#444444', wrap=True)

png_path = OUT / 'panel_vs_meropenem_jupyter.png'
svg_path = OUT / 'panel_vs_meropenem_jupyter.svg'
fig.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(svg_path, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved PNG:', png_path)
print('Saved SVG:', svg_path)

## How to read it

The green cells show matching categories and the amber cells show mismatches. Percentages are calculated within each AST row. The figure describes this selected public dataset; it is not a clinical test evaluation.